# Full Model CPT with 4-Bit Quantization
## Qwen2.5-7B - Complete Model Training (Not LoRA)

- Trains the FULL model (not adapters)
- Uses 4-bit quantization (model: 15GB → 3GB)
- Fits on 31.84GB GPU
- Same quality as full precision

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
import json
from pathlib import Path
from datetime import datetime
from typing import Dict, List

print("\n" + "="*80)
print("FULL MODEL CPT - 4-BIT QUANTIZATION")
print("="*80)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("="*80 + "\n")

In [ ]:
# ============ CONFIGURATION ============
config = {
    "model_name": "Qwen/Qwen2.5-7B",
    "train_file": "augmented_output/train.jsonl",
    "eval_file": "augmented_output/eval.jsonl",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-5,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "max_seq_length": 512,
    "bf16": True,
    "output_dir": "medical_qwen_cpt_4bit",
    "save_steps": 100,
    "eval_steps": 50,
    "logging_steps": 5,
    "dataloader_num_workers": 0,
    "dataloader_pin_memory": False,
    "seed": 42,
}

print("Configuration:")
print("="*70)
for k, v in config.items():
    print(f"{k:.<50} {v}")
print("="*70)
print(f"Effective batch size: {config['per_device_train_batch_size'] * config['gradient_accumulation_steps']}")
print("="*70 + "\n")

In [ ]:
# ============ LOAD DATA ============
def load_jsonl(file_path: str) -> List[Dict]:
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            try:
                data.append(json.loads(line))
            except:
                pass
    return data

print("Loading data...")
train_data = load_jsonl(config["train_file"])
eval_data = load_jsonl(config["eval_file"])

print(f"✅ Train: {len(train_data):,} chunks")
print(f"✅ Eval:  {len(eval_data):,} chunks\n")

In [ ]:
# ============ LOAD TOKENIZER ============
from transformers import AutoTokenizer

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    config["model_name"],
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded\n")

In [ ]:
# ============ LOAD MODEL IN 4-BIT ============
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

print("Loading model in 4-bit quantization...")
print("(This reduces 15GB model → 3GB)\n")

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Load model
model = AutoModelForCausalLM.from_pretrained(
    config["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Enable gradient checkpointing
model.gradient_checkpointing_enable()

print(f"✅ Model loaded")
num_params = sum(p.numel() for p in model.parameters())
print(f"   Parameters: {num_params/1e9:.2f}B")
print(f"   Quantization: 4-bit")
print(f"   Gradient checkpointing: Enabled")

allocated = torch.cuda.memory_allocated(0) / 1e9
print(f"   GPU memory: {allocated:.2f} GB\n")

if allocated > 20:
    print("⚠️  WARNING: Model using >20GB - training may still OOM!")
else:
    print("✅ Good: Model uses <20GB, training should fit!\n")

In [ ]:
# ============ TOKENIZE DATASETS ============
from datasets import Dataset

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=config["max_seq_length"],
        padding="max_length",
    )

print("Tokenizing datasets...")
train_dataset = Dataset.from_dict({"text": [c["text"] for c in train_data]})
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
)

eval_dataset = Dataset.from_dict({"text": [c["text"] for c in eval_data]})
eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=100,
    remove_columns=["text"],
)

print(f"✅ Train: {len(train_dataset):,} samples")
print(f"✅ Eval: {len(eval_dataset):,} samples\n")

In [ ]:
# ============ SETUP TRAINING ============
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir=config["output_dir"],
    num_train_epochs=config["num_train_epochs"],
    per_device_train_batch_size=config["per_device_train_batch_size"],
    per_device_eval_batch_size=config["per_device_eval_batch_size"],
    gradient_accumulation_steps=config["gradient_accumulation_steps"],
    learning_rate=config["learning_rate"],
    warmup_steps=config["warmup_steps"],
    weight_decay=config["weight_decay"],
    max_grad_norm=config["max_grad_norm"],
    bf16=config["bf16"],
    save_strategy="steps",
    save_steps=config["save_steps"],
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=config["eval_steps"],
    logging_dir=config["logging_dir"],
    logging_steps=config["logging_steps"],
    seed=config["seed"],
    dataloader_num_workers=config["dataloader_num_workers"],
    dataloader_pin_memory=config["dataloader_pin_memory"],
    load_best_model_at_end=True,
    greater_is_better=False,
    push_to_hub=False,
    report_to=["tensorboard"],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("✅ Training setup complete\n")
print("="*80)
print("🚀 READY TO TRAIN FULL MODEL")
print("="*80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Expected duration: 10-18 hours")
print(f"Output: {config['output_dir']}/")
print("="*80 + "\n")

In [ ]:
# ============ START TRAINING ============
print(f"\n🚀 Starting training...\n")

try:
    train_result = trainer.train()
    print(f"\n✅ Training complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Training loss: {train_result.training_loss:.4f}\n")
except KeyboardInterrupt:
    print("\n⏹️  Training interrupted")
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ============ EVALUATE ============
print("\nEvaluating...")
eval_results = trainer.evaluate()

print("\nResults:")
for key, value in sorted(eval_results.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
print()

In [ ]:
# ============ SAVE MODEL ============
best_model_path = Path(config["output_dir"]) / "best_model"
best_model_path.mkdir(parents=True, exist_ok=True)

print(f"Saving model...")
trainer.model.save_pretrained(str(best_model_path))
tokenizer.save_pretrained(str(best_model_path))

model_size = sum(f.stat().st_size for f in best_model_path.glob('**/*')) / 1e9
print(f"✅ Model saved to {best_model_path}")
print(f"   Size: {model_size:.2f}GB\n")

print("="*80)
print("✅ FULL MODEL TRAINING COMPLETE")
print("="*80)